<a href="https://colab.research.google.com/github/AKsToolBox/AKsToolBox/blob/main/Hermes__blated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
set -e
mkdir -p /content/llama_bin

TAG=$(curl -fsSL https://github.com/ggml-org/llama.cpp/releases/download/v1.0.0/nightly-tag.txt | tr -d '[:space:]')
TAG=${TAG:-b12359}

BASE_URL="https://github.com/ggml-org/llama.cpp/releases/download/${TAG}"
wget -q --show-progress --progress=bar:force:noscroll "${BASE_URL}/llama-${TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz" -O /tmp/llama-bin.tar.gz
wget -q --show-progress --progress=bar:force:noscroll "${BASE_URL}/cudart-llama-${TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz" -O /tmp/llama-cudart.tar.gz

tar -xzf /tmp/llama-bin.tar.gz -C /content/llama_bin --strip-components=1
tar -xzf /tmp/llama-cudart.tar.gz -C /content/llama_bin --strip-components=1
chmod +x /content/llama_bin/llama-server

LD_LIBRARY_PATH=/content/llama_bin:/usr/lib64-nvidia /content/llama_bin/llama-server --list-devices

Available devices:
  CUDA0: Tesla T4 (14912 MiB, 14807 MiB free)


/tmp/llama-bin.tar. 100%[===================>] 189.99M  93.6MB/s    in 2.0s    
/tmp/llama-cudart.t 100%[===================>] 566.84M  38.1MB/s    in 5.4s    


In [ ]:
from google.colab import drive
import subprocess
import os
import time

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/pentest_foundry/Hermes-3-Llama-3.1-8B-lorablated-Q6_K.gguf"
server_path = "/content/llama_bin/llama-server"

if not os.path.exists(server_path):
    raise FileNotFoundError(f"Server binary not found at {server_path}. Run Cell 1 first.")
if not os.path.exists(model_path):
    raise FileNotFoundError(f"GGUF model not found at {model_path}. Check your Drive folder path.")

# Stop any existing llama-server instance before starting a new one
subprocess.run(["pkill", "-f", "llama-server"], check=False)

env = {**os.environ, "LD_LIBRARY_PATH": "/content/llama_bin:/usr/lib64-nvidia"}
log_path = "/content/llama_server.log"

with open(log_path, "w") as log_file:
    process = subprocess.Popen([
        server_path,
        "-m", model_path,
        "-ngl", "99",
        "-fa", "on",
        "--jinja",
        "--ctx-size", "16384",
        "--temp", "0.7",
        "--min-p", "0.05",
        "--host", "0.0.0.0",
        "--port", "8081"
    ], stdout=log_file, stderr=subprocess.STDOUT, env=env)

print("Loading Hermes 3 Q6_K onto Tesla T4...")
for _ in range(60):
    if process.poll() is not None:
        print("Server exited unexpectedly:")
        break
    if os.path.exists(log_path):
        with open(log_path) as f:
            logs = f.read()
        if "listening" in logs.lower():
            print("Server is live on http://0.0.0.0:8081")
            break
    time.sleep(2)

!tail -n 12 /content/llama_server.log

Loading Hermes 3 Q6_K onto Tesla T4...
0.00.166.553 I srv  llama_server: initializing ...
0.00.166.649 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
0.00.167.580 W srv  server_setup: security: no API key is set and CORS allows all origins (see https://github.com/ggml-org/llama.cpp/pull/25655)
0.00.169.261 I srv    load_model: loading model '/content/drive/MyDrive/pentest_foundry/Hermes-3-Llama-3.1-8B-lorablated-Q6_K.gguf'
1.24.834.146 I cmn          init: llama threadpool init, n_threads = 1


In [ ]:
import subprocess
import re
import time
import os

if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

subprocess.run(["pkill", "-f", "cloudflared"], check=False)

tunnel_log = "/content/cloudflared.log"
with open(tunnel_log, "w") as f:
    tunnel_process = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://localhost:8081"],
        stdout=f,
        stderr=subprocess.STDOUT
    )

tunnel_url = None
for _ in range(30):
    if os.path.exists(tunnel_log):
        with open(tunnel_log) as f:
            match = re.search(r"(https://[a-zA-Z0-9-]+\.trycloudflare\.com)", f.read())
            if match:
                tunnel_url = match.group(1)
                break
    time.sleep(0.5)

if tunnel_url:
    print(f"OpenAI-compatible Base URL: {tunnel_url}/v1")
else:
    print("Could not retrieve Cloudflare Tunnel URL. Check logs:")
    !tail -n 15 /content/cloudflared.log

OpenAI-compatible Base URL: https://observations-extended-freebsd-secret.trycloudflare.com/v1


In [ ]:
import requests

# Retrieve the generated tunnel URL from the previous cell variable
test_url = f"{tunnel_url}/v1/models"
print(f"Verifying Cloudflare Tunnel by requesting: {test_url}")

try:
    response = requests.get(test_url, timeout=10)
    if response.status_code == 200:
        print("\nSuccess! Cloudflare Tunnel is working and connected to llama-server.")
        print("Response:", response.json())
    else:
        print(f"\nFailed to connect. HTTP Status Code: {response.status_code}")
        print("Response text:", response.text)
except Exception as e:
    print(f"\nAn error occurred while testing the connection: {e}")
    print("Please make sure llama-server has finished loading and is listening on port 8081.")

Verifying Cloudflare Tunnel by requesting: https://observations-extended-freebsd-secret.trycloudflare.com/v1/models

An error occurred while testing the connection: HTTPSConnectionPool(host='observations-extended-freebsd-secret.trycloudflare.com', port=443): Max retries exceeded with url: /v1/models (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x79f35eccef90>: Failed to resolve 'observations-extended-freebsd-secret.trycloudflare.com' ([Errno -2] Name or service not known)"))
Please make sure llama-server has finished loading and is listening on port 8081.
